# Week 1 — 파이프라인 잠그기 (steering vector 실험)

> 설계 v2 §8의 1주차 작업을 그대로 실행하는 노트북.
> **목표 산출물**: B0 두 숫자(eval_v1 / eval_toxigen_v1) + FN subset 크기 + v_AB sanity 한 줄.

### 1주차 체크리스트
- [ ] eval_v1 leakage check (∩ Cell A/B/C = 0, ∩ HateXplain = 0)
- [ ] eval_toxigen_v1 구축 (13그룹 stratified ~1,500)
- [ ] `metrics.py` 픽스 (evaluate 단일 함수)
- [ ] probe(**HateXplain** 학습) → scaler+clf 저장
- [ ] B0 baseline 두 평가셋 측정 → `b0_baseline.json`
- [ ] FN subset 인덱스 저장 (두 평가셋)
- [ ] v_AB 한 레이어 sanity hook → F1이 B0 대비 움직이는지 확인

### 핵심 불변식 (절대 안 깸)
1. **풀링/위치 일관**: 모든 hidden은 **left-padding + 마지막 토큰(`[:, -1, :]`)**으로 읽는다. 주입도 같은 위치.
2. **probe는 no-steering HateXplain으로 한 번만 학습**, 이후 predict만.
3. 평가셋·split·라벨매핑·메트릭은 한 번 잠그면 끝까지 안 바꾼다.

## 0. 설정 & 경로

In [1]:
# !pip -q install transformers datasets scikit-learn pandas numpy

import torch
import os, json, hashlib, re
import numpy as np, pandas as pd

# ---- 경로 (본인 환경에 맞게만 바꾸세요) ----
class CFG:
    MODEL = "meta-llama/Llama-3.2-3B"

    # Latent Hatred 2000건 단일 csv
    EVAL_LATENT_CSV      = "data/eval/eval_latent_v2.csv"
    # 컬럼명이 다르면 아래 매핑만 수정
    EVAL_LATENT_TEXT_COL     = "text"
    EVAL_LATENT_LABEL_COL    = "label"          # hate=1, non-hate=0

    # vector 추출용 Cell 데이터 (벡터 sanity에 사용)
    CELL_C_CSV           = "cell_c_test_final.csv"            # idx, cell_a, cell_c_generated, cell_c_modified
    CELL_B_CSV           = "cell_bbb_domain_v10_256_revised.csv"  # text_clean, ..., rewrite, ...

    # probe 학습 = v_harm 추출에 쓴 것과 *동일* HateXplain split
    #   있으면 이 csv(text,label) 사용, 없으면 HF에서 만들어 저장
    HATEXPLAIN_TRAIN_CSV = "data/hatexplain_train.csv"

    OUT_DIR              = "results"
    DATA_EVAL_DIR        = "data/eval"
    SRC_DIR              = "src/eval"

    # sanity hook 설정
    SANITY_LAYER         = 20      # late, harm-dominant 구간
    SANITY_ALPHA         = 4.0

    BATCH                = 16
    MAX_LEN              = 128
    SEED                 = 20260528

for d in [CFG.OUT_DIR, CFG.DATA_EVAL_DIR, CFG.SRC_DIR]:
    os.makedirs(d, exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
np.random.seed(CFG.SEED)
print("device:", device)

device: cuda


In [2]:
# HF 로그인 (Llama / ToxiGen / HateXplain 모두 gated일 수 있음)
# from huggingface_hub import login; login("hf_xxx")

from transformers import AutoModelForCausalLM, AutoTokenizer

tok = AutoTokenizer.from_pretrained(CFG.MODEL)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
tok.padding_side = "left"   # 마지막 토큰이 항상 위치 -1 → 주입/probe 위치 일관

model = AutoModelForCausalLM.from_pretrained(
    CFG.MODEL, torch_dtype=torch.float16, output_hidden_states=True
).to(device).eval()

N_LAYERS = model.config.num_hidden_layers          # 28
HIDDEN   = model.config.hidden_size                # 3072
print(f"layers={N_LAYERS}  hidden={HIDDEN}  (hidden_states 길이 = {N_LAYERS+1})")

The following generation flags are not valid and may be ignored: ['output_hidden_states']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['output_hidden_states']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

layers=28  hidden=3072  (hidden_states 길이 = 29)


## 1. 공용 함수 — hidden 추출 / metrics / steering hook

`extract_hidden`은 left-padding + 마지막 토큰만 읽는다.
`keep_layers=None`이면 마지막 레이어만 `(N, 3072)`, 리스트면 `(N, len, 3072)`.

In [3]:
@torch.no_grad()
def extract_hidden(texts, keep_layers=None, batch_size=CFG.BATCH):
    '''left-pad + 마지막 토큰 hidden.
    keep_layers=None -> (N, HIDDEN) (마지막 레이어)
    keep_layers=[..] -> (N, len(keep_layers), HIDDEN)'''
    feats = []
    for i in range(0, len(texts), batch_size):
        batch = [str(t) for t in texts[i:i+batch_size]]
        enc = tok(batch, return_tensors="pt", padding=True,
                  truncation=True, max_length=CFG.MAX_LEN).to(device)
        hs = model(**enc).hidden_states            # tuple(N_LAYERS+1) of (B, T, H)
        if keep_layers is None:
            feats.append(hs[-1][:, -1, :].float().cpu().numpy())          # (B, H)
        else:
            stk = np.stack([hs[L][:, -1, :].float().cpu().numpy()
                            for L in keep_layers], axis=1)                # (B, len, H)
            feats.append(stk)
    return np.concatenate(feats, axis=0)


class SteeringHook:
    def __init__(self, vec, alpha):
        self.v = torch.tensor(vec, dtype=torch.float16, device=device)
        self.a = float(alpha); self.h = None
    def _fn(self, m, i, o):
        # transformers 버전에 따라 layer 출력이 tuple 또는 tensor
        if isinstance(o, tuple):
            o[0][:, -1, :] = o[0][:, -1, :] + self.a * self.v
            return o
        else:
            o[:, -1, :] = o[:, -1, :] + self.a * self.v
            return o
    def attach(self, layer): self.h = layer.register_forward_hook(self._fn); return self
    def detach(self):
        if self.h is not None: self.h.remove(); self.h = None

def norm(text):  # leakage check용 정규화
    return re.sub(r"\s+", " ", re.sub(r"[^a-z0-9 ]", "", str(text).lower())).strip()

In [4]:
%%writefile src/eval/metrics.py
from sklearn.metrics import f1_score, recall_score
HATE, NON_HATE = 1, 0

def evaluate(preds, labels):
    '''모든 실험은 오직 이 함수만 호출한다.'''
    return {"macro_f1":   f1_score(labels, preds, average="macro"),
            "hate_recall": recall_score(labels, preds, pos_label=HATE)}

def evaluate_by_group(preds, labels, groups):
    out = {}
    for g in set(groups):
        idx = [k for k, gi in enumerate(groups) if gi == g]
        l = [labels[k] for k in idx]; p = [preds[k] for k in idx]
        out[g] = {"hate_recall": recall_score(l, p, pos_label=HATE) if HATE in l else None,
                  "n": len(p)}
    return out

Overwriting src/eval/metrics.py


In [5]:
import importlib, sys
sys.path.insert(0, "src/eval")
import metrics; importlib.reload(metrics)
from metrics import evaluate, evaluate_by_group, HATE

# 단위 테스트
assert abs(evaluate([1,0,1,0],[1,0,1,0])["macro_f1"] - 1.0) < 1e-9
print("metrics.py OK")

metrics.py OK


## 2. 데이터 로드 & Cell A/B/C 정렬 (sanity용 v_AB)

In [6]:
# eval_v1 (이미 구축됨)
ev = pd.read_csv(CFG.EVAL_LATENT_CSV)
ev = ev.rename(columns={CFG.EVAL_LATENT_TEXT_COL: "text", CFG.EVAL_LATENT_LABEL_COL: "label"})
ev["label"] = ev["label"].astype(str).str.strip().str.lower().map({"hate": 1, "non-hate": 0}).astype(int)
eval_latent_texts    = ev["text"].tolist()
eval_latent_labels  = ev["label"].to_numpy()
print(f"eval_latent_v2: {len(ev)}건, label 분포 {np.bincount(eval_latent_labels)}")

# Cell A/B/C 정렬 (idx 같은 A/B/C 한 쌍만, PASS 결측 제외)
dfc = pd.read_csv(CFG.CELL_C_CSV)
dfb = pd.read_csv(CFG.CELL_B_CSV)
tri = (dfc[["idx", "cell_a", "cell_c_modified"]]
       .rename(columns={"cell_a": "A", "cell_c_modified": "C"}))
tri = tri.merge(
    dfb[["text_clean", "generated_text"]].rename(columns={"text_clean": "A", "generated_text": "B"}),
    on="A", how="inner")
tri = tri.dropna(subset=["A", "B", "C"])
tri = tri[(tri["B"].astype(str).str.len() > 0) & (tri["C"].astype(str).str.len() > 0)]
A_texts, B_texts, C_texts = tri["A"].tolist(), tri["B"].tolist(), tri["C"].tolist()
print(f"정렬된 PASS triple: {len(tri)}건  (기대 ~238)")

eval_latent_v2: 2000건, label 분포 [1000 1000]
정렬된 PASS triple: 256건  (기대 ~238)


## 3. eval_v1 leakage check

eval_v1 텍스트가 (a) Cell A/B/C, (b) HateXplain 어디와도 겹치지 않는지 **숫자로** 확인.

In [7]:
# HateXplain 텍스트 풀 로드 (겹침 검사 + probe 학습 양쪽에 쓰임)
def load_hatexplain():
    '''본인이 v_harm 추출에 쓴 split이 있으면 그걸, 없으면 HF에서 majority-vote로 생성.'''
    if os.path.exists(CFG.HATEXPLAIN_TRAIN_CSV):
        df = pd.read_csv(CFG.HATEXPLAIN_TRAIN_CSV)
        print("HateXplain: 기존 split 파일 사용 ->", CFG.HATEXPLAIN_TRAIN_CSV)
        return df
    from datasets import load_dataset
    ds = load_dataset("hatexplain", split="train")
    rows = []
    for r in ds:
        text = " ".join(r["post_tokens"])
        labs = r["annotators"]["label"]            # 0 hatespeech / 1 normal / 2 offensive (데이터셋 정의)
        maj = max(set(labs), key=labs.count)
        # hate = hatespeech(0) or offensive(2) -> 1, normal(1) -> 0
        rows.append({"text": text, "label": 0 if maj == 1 else 1})
    df = pd.DataFrame(rows)
    df.to_csv(CFG.HATEXPLAIN_TRAIN_CSV, index=False)
    print("HateXplain: HF에서 생성·저장 ->", CFG.HATEXPLAIN_TRAIN_CSV,
          "  ⚠️ v_harm 추출 split과 동일한지 반드시 확인")
    return df

hx = load_hatexplain()
print(f"HateXplain: {len(hx)}건, label 분포 {np.bincount(hx['label'].astype(int))}")

HateXplain: 기존 split 파일 사용 -> data/hatexplain_train.csv
HateXplain: 15383건, label 분포 [6251 9132]


In [8]:
ev_norm = set(ev["text"].map(norm))
cell_norm = set(map(norm, A_texts + B_texts + C_texts))
hx_norm = set(hx["text"].map(norm))

ov_cell = len(ev_norm & cell_norm)
ov_hx   = len(ev_norm & hx_norm)
print(f"eval_latent_v2 ∩ Cell A/B/C : {ov_cell} 건")
print(f"eval_latent_v2 ∩ HateXplain : {ov_hx} 건")

with open(os.path.join(CFG.DATA_EVAL_DIR, "leakage_check.md"), "w") as f:
    f.write("# leakage check\n\n")
    f.write(f"- eval_latent_v2 ∩ Cell A/B/C: **{ov_cell}** 건\n")
    f.write(f"- eval_latent_v2 ∩ HateXplain: **{ov_hx}** 건\n")
print("→ data/eval/leakage_check.md 저장")
if ov_cell or ov_hx:
    print("⚠️ 겹침 발견 — 해당 행을 eval_v1에서 제거 후 다시 진행하세요.")

eval_latent_v2 ∩ Cell A/B/C : 0 건
eval_latent_v2 ∩ HateXplain : 0 건
→ data/eval/leakage_check.md 저장


## 4. eval_toxigen_v1 구축

`toxigen/toxigen-data`의 **annotated** split만. `toxicity_ai ≥ 4`→hate, `≤ 2`→non-hate, `==3`→제외.
그룹 × label 셀당 동수로 stratified, 부족 그룹은 보충하지 않고 그대로 둔다.

In [9]:
from datasets import load_dataset

tg = load_dataset("toxigen/toxigen-data", name="annotated")
tg = pd.concat([tg[s].to_pandas() for s in tg.keys()], ignore_index=True)
tg.columns = [c.lower() for c in tg.columns]
print("toxigen annotated rows:", len(tg))

# 라벨 매핑
tg = tg[tg["toxicity_ai"] != 3].copy()
tg["label"] = (tg["toxicity_ai"] >= 4).astype(int)

# ── target_group(free-text) → 표준 13 카테고리 (구체적인 것 먼저) ──
RULES = [
    ("chinese",         ["chinese"]),
    ("asian",           ["asian"]),
    ("mexican",         ["mexican"]),
    ("latino",          ["latino","latina","hispanic"]),
    ("muslim",          ["muslim","islam"]),
    ("jewish",          ["jewish","jew"]),
    ("black",           ["black","african"]),
    ("lgbtq",           ["lgbtq","lgbt","gay","lesbian","queer","bisexual","trans"]),
    ("women",           ["women","woman","female"]),
    ("middle_east",     ["middle eastern","middle-eastern","middle east","middle_east"]),
    ("native_american", ["native american","native-american","indigenous","native_american"]),
    ("mental_dis",      ["mental disab","mental ill","mental health","cognitive","mental_dis"]),
    ("physical_dis",    ["physical disab","disab","physical_dis"]),
]
def to13(g):
    s = str(g).lower()
    for canon, keys in RULES:
        if any(k in s for k in keys): return canon
    return "OTHER"

tg["group13"] = tg["target_group"].map(to13)
unmapped = sorted(tg.loc[tg.group13=="OTHER","target_group"].unique())
print("UNMAPPED(OTHER) 원본 라벨:", unmapped)        # 비어 있어야 정상
tg = tg[tg["group13"] != "OTHER"].copy()             # OTHER는 버림

# ── 13 그룹 × label 셀당 60건 stratified ──
PER_GROUP_PER_LABEL = 60
parts = []
for g, gdf in tg.groupby("group13"):
    for lab in [0, 1]:
        sub = gdf[gdf["label"] == lab]
        take = min(PER_GROUP_PER_LABEL, len(sub))
        if take: parts.append(sub.sample(take, random_state=CFG.SEED))
tox = pd.concat(parts, ignore_index=True)

# 컬럼 정리 — target_group은 표준 13값, 원본은 _raw로 보존
tox = tox.rename(columns={"group13": "target_group", "target_group": "target_group_raw"})
tox = tox[["text", "label", "target_group", "target_group_raw", "toxicity_ai"]]
tox["source"] = "toxigen_humanval"
tox.insert(0, "id", [f"TG{i:05d}" for i in range(len(tox))])

# HateXplain leakage guard
tox = tox[~tox["text"].map(norm).isin(hx_norm)].reset_index(drop=True)

tox.to_csv(os.path.join(CFG.DATA_EVAL_DIR, "eval_toxigen_v1.csv"), index=False)
print(f"eval_toxigen_v1: {len(tox)}건, 그룹 {tox['target_group'].nunique()}개, label {np.bincount(tox['label'])}")
print(tox.groupby("target_group").size())
toxigen_texts  = tox["text"].tolist()
toxigen_labels = tox["label"].to_numpy()
toxigen_groups = tox["target_group"].tolist()

toxigen annotated rows: 9900
UNMAPPED(OTHER) 원본 라벨: []
eval_toxigen_v1: 1560건, 그룹 13개, label [780 780]
target_group
asian              120
black              120
chinese            120
jewish             120
latino             120
lgbtq              120
mental_dis         120
mexican            120
middle_east        120
muslim             120
native_american    120
physical_dis       120
women              120
dtype: int64


## 5. probe (HateXplain) 학습 & 저장

In [10]:
import time, numpy as np, torch
from tqdm.auto import tqdm
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
import joblib

print("torch.cuda.is_available():", torch.cuda.is_available())
print("model device:", next(model.parameters()).device, "| dtype:", next(model.parameters()).dtype)

@torch.inference_mode()
def extract_last_fast(texts, batch_size=16, max_len=CFG.MAX_LEN, desc="extract"):
    texts = [str(t) for t in texts]
    order = sorted(range(len(texts)), key=lambda i: len(texts[i]))
    inv = np.argsort(order)
    out = []
    for s in tqdm(range(0, len(order), batch_size), desc=desc):
        idx = order[s:s+batch_size]
        enc = tok([texts[i] for i in idx], return_tensors="pt",
                  padding=True, truncation=True, max_length=max_len).to(device)
        h = model.model(**enc,
                        use_cache=False,
                        output_hidden_states=False   # ← 29레이어 안 만듦 (핵심)
                        ).last_hidden_state[:, -1, :]
        out.append(h.float().cpu().numpy())
        del enc, h
    return np.concatenate(out, 0)[inv]                # 원래 순서로 복원

print("HateXplain hidden 추출 중...", len(hx))
from sklearn.model_selection import train_test_split

# 90% 학습 / 10% sanity held-out (stratified, 고정 seed)
hx_train, hx_hold = train_test_split(hx, test_size=0.1,
                                     stratify=hx["label"].astype(int), random_state=CFG.SEED)
print(f"hx_train={len(hx_train)}  hx_hold={len(hx_hold)}")

t0 = time.time()
X_tr = extract_last_fast(hx_train["text"].tolist())
y_tr = hx_train["label"].astype(int).to_numpy()
print(f"추출 완료: {X_tr.shape}  ({time.time()-t0:.1f}s)")

scaler = StandardScaler().fit(X_tr)
clf = LogisticRegression(C=1.0, max_iter=1000).fit(scaler.transform(X_tr), y_tr)
joblib.dump({"scaler": scaler, "clf": clf}, os.path.join(CFG.OUT_DIR, "probe.pkl"))
print("probe.pkl(90%) 저장. train acc =", clf.score(scaler.transform(X_tr), y_tr))

# sanity_hatexplain — held-out 10%에서 균형 ~500건
nph = 250
s1 = hx_hold[hx_hold.label==1].sample(min(nph, int((hx_hold.label==1).sum())), random_state=CFG.SEED)
s0 = hx_hold[hx_hold.label==0].sample(min(nph, int((hx_hold.label==0).sum())), random_state=CFG.SEED)
sanity = pd.concat([s1, s0]).reset_index(drop=True); sanity["source"] = "hatexplain_heldout"
sanity[["text","label","source"]].to_csv(os.path.join(CFG.DATA_EVAL_DIR, "sanity_hatexplain.csv"), index=False)
print(f"sanity_hatexplain={len(sanity)}건 저장, label {np.bincount(sanity['label'].astype(int))}")

torch.cuda.is_available(): True
model device: cuda:0 | dtype: torch.float16
HateXplain hidden 추출 중... 15383
hx_train=13844  hx_hold=1539


extract:   0%|          | 0/866 [00:00<?, ?it/s]

추출 완료: (13844, 3072)  (135.0s)
probe.pkl(90%) 저장. train acc = 0.8661514013290956
sanity_hatexplain=500건 저장, label [250 250]


## 6. B0 (No steering) baseline — 두 평가셋

In [11]:
def predict(texts):
    X = extract_last_fast(texts)             # no-steering, last-token
    return clf.predict(scaler.transform(X))

pred_latent  = predict(eval_latent_texts);   b0_latent  = evaluate(pred_latent, eval_latent_labels)
pred_tg = predict(toxigen_texts);   b0_tg = evaluate(pred_tg, toxigen_labels)
sa_texts = sanity["text"].tolist(); sa_labels = sanity["label"].astype(int).to_numpy()
pred_sa = predict(sa_texts);        b0_sa = evaluate(pred_sa, sa_labels)

b0 = {"model": CFG.MODEL, "probe": "logistic_regression", "layer": -1,
  "results": {
    "eval_latent_v2":           {**b0_latent, "n": int(len(eval_latent_labels))},
    "eval_toxigen_v1":   {**b0_tg, "n": int(len(toxigen_labels))},
    "sanity_hatexplain": {**b0_sa, "n": int(len(sa_labels))},
  }}
json.dump(b0, open(os.path.join(CFG.OUT_DIR, "b0_baseline.json"), "w"), indent=2)
print("B0 eval_latent_v2      :", b0_latent)
print("B0 eval_toxigen_v1:", b0_tg)
print("B0 sanity_hatexplain:", b0_sa)

extract:   0%|          | 0/125 [00:00<?, ?it/s]

extract:   0%|          | 0/98 [00:00<?, ?it/s]

extract:   0%|          | 0/32 [00:00<?, ?it/s]

B0 eval_latent_v2      : {'macro_f1': 0.5902772490673548, 'hate_recall': 0.549}
B0 eval_toxigen_v1: {'macro_f1': 0.5853791598472449, 'hate_recall': 0.4230769230769231}
B0 sanity_hatexplain: {'macro_f1': 0.6908871938980329, 'hate_recall': 0.752}


## 7. FN subset 인덱스 저장 (B0가 놓친 hate)

In [12]:
def save_fn_subset(pred, labels, name):
    fn = np.where((labels == HATE) & (pred == 0))[0]
    np.save(os.path.join(CFG.OUT_DIR, f"fn_subset_{name}.npy"), fn)
    print(f"{name}: FN subset {len(fn)}건 / hate {int((labels==HATE).sum())}건 저장")
    return fn

fn_latent = save_fn_subset(pred_latent, eval_latent_labels, "eval_latent_v2")
fn_tg = save_fn_subset(pred_tg, toxigen_labels, "eval_toxigen_v1")
# 이 인덱스는 이후 sweep에서 절대 재생성하지 않는다.

eval_latent_v2: FN subset 451건 / hate 1000건 저장
eval_toxigen_v1: FN subset 450건 / hate 780건 저장


## 8. v_AB 한 레이어 sanity hook

v_AB(target 축)를 한 레이어에 주입했을 때 eval_v1 macro F1이 B0 대비 **어떻게든 움직이면** 파이프라인 OK.
(올라가든 내려가든 상관없음. 똑같으면 hook이 안 걸린 것.)

In [13]:
# v_AB 추출 (전 레이어) — 이후 주차에서 재사용하도록 저장
print("Cell A/B hidden 추출 중...")
h_A = extract_hidden(A_texts, keep_layers=list(range(N_LAYERS+1)))   # (n,29,H)
h_B = extract_hidden(B_texts, keep_layers=list(range(N_LAYERS+1)))

def steering_vector(h_src, h_tgt):
    d = h_src.mean(0) - h_tgt.mean(0)
    return d / (np.linalg.norm(d, axis=1, keepdims=True) + 1e-8)

v_AB = steering_vector(h_A, h_B)             # (29, H)
np.save(os.path.join(CFG.OUT_DIR, "v_AB.npy"), v_AB)
print("v_AB.npy 저장:", v_AB.shape)

Cell A/B hidden 추출 중...
v_AB.npy 저장: (29, 3072)


In [14]:
L, a = CFG.SANITY_LAYER, CFG.SANITY_ALPHA
hook = SteeringHook(v_AB[L], a).attach(model.model.layers[L])
try:
    X_steered = extract_hidden(eval_latent_texts)       # 스티어링 적용 상태
finally:
    hook.detach()

pred_s = clf.predict(scaler.transform(X_steered))
m_s = evaluate(pred_s, eval_latent_labels)
delta = m_s["macro_f1"] - b0_latent["macro_f1"]
print(f"sanity (v_AB, L={L}, α={a}): macro F1 = {m_s['macro_f1']:.4f}  (B0 대비 {delta:+.4f})")
print("PASS — hook 동작 확인" if abs(delta) > 1e-6 else "⚠️ B0와 동일 → hook 미적용, 디버깅 필요")

sanity (v_AB, L=20, α=4.0): macro F1 = 0.5933  (B0 대비 +0.0030)
PASS — hook 동작 확인


## 9. 1주차 보고 3줄

In [15]:
print("="*60)
print(f"1. eval_v2 (Latent Hatred, n={len(eval_latent_labels)}) 확정, B0 macro F1 = {b0_latent['macro_f1']:.4f}")
print(f"2. eval_toxigen_v1 (n={len(toxigen_labels)}, {tox["target_group"].nunique()}그룹) 확정, B0 macro F1 = {b0_tg['macro_f1']:.4f}")
print(f"3. v_AB sanity: L={CFG.SANITY_LAYER}, α={CFG.SANITY_ALPHA} 에서 macro F1 {delta:+.4f} 이동 → 파이프라인 동작 확인")
print(f"   (FN subset: eval_v1 {len(fn_latent)}건 / eval_toxigen_v1 {len(fn_tg)}건)")
print("="*60)

1. eval_v2 (Latent Hatred, n=2000) 확정, B0 macro F1 = 0.5903
2. eval_toxigen_v1 (n=1560, 13그룹) 확정, B0 macro F1 = 0.5854
3. v_AB sanity: L=20, α=4.0 에서 macro F1 +0.0030 이동 → 파이프라인 동작 확인
   (FN subset: eval_v1 451건 / eval_toxigen_v1 450건)


In [16]:
import json
print(json.load(open("results/b0_baseline.json"))["results"]["sanity_hatexplain"])

{'macro_f1': 0.6908871938980329, 'hate_recall': 0.752, 'n': 500}
